# DhikrSpeech — end-to-end pipeline

The whole training pipeline in one notebook: from the raw recordings on Drive to the quantised
TensorFlow Lite model the Android app ships, and the evidence that it works as a *counter* rather
than as a classifier.

Run **Setup** once, then work through the stages in order:

1. **Dataset** — inspect and validate the recordings; who spoke them, and is there enough of them.
2. **Preprocessing** — condition to 16 kHz mono, freeze a **speaker-safe** train/val/test split,
   write the manifest.
3. **Training** — train the DS-CNN (TensorBoard, checkpoints, resume).
4. **Evaluation** — clip metrics, confusion matrix, ROC, and what kind of audio produces false
   positives.
5. **Export** — SavedModel + TFLite variants, benchmarked and verified.
6. **Streaming** — the deployed shape: sliding windows, event counting, **false activations per
   hour**, threshold calibration, the Android contract, and a production-readiness verdict.
7. **Experiment** *(optional)* — one model per dhikr, or one model over all of them?

Stages hand off through files on Drive (manifest, checkpoints, reports), not in-memory state, so
after Setup you can also jump to and run a single stage on its own.

**Clip accuracy is not the deliverable.** A model can score well in stage 4 and still count a
conversation as dhikr; stage 6 is what measures the difference, and its verdict is the one to read.

**Runtime → Change runtime type → GPU** before training, or it falls back to CPU.

In [26]:
#@title Setup — mount Drive, locate the project, install what is missing
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MahmoudMabrok/SaloAleh.git"
REPO_BRANCH = os.environ.get("DHIKR_BRANCH", "main")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def sync_clone(target: Path) -> None:
    """Pull the latest code into an existing clone.

    Without this a runtime that cloned the repository earlier keeps running that
    old copy for the rest of the session, so fixes never arrive.
    """
    subprocess.run(
        ["git", "-C", str(target), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True
    )
    subprocess.run(
        ["git", "-C", str(target), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
    )


def find_project_root() -> Path:
    """The folder that holds src/ and configs/ — cloned or updated as needed."""
    candidates = [Path(p) for p in [
        os.environ.get("DHIKR_PROJECT_ROOT", ""),
        "/content/DhikrSpeech",
        "/content/SaloAleh/DhikrSpeech",
        "/content/drive/MyDrive/DhikrSpeech",
    ] if p]
    candidates += [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "config.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            # Only ever update the throwaway clone this notebook created. A repo
            # you checked out yourself is left alone - resetting it would discard
            # whatever branch and local edits you are working on.
            if IN_COLAB and candidate.parent == Path("/content/SaloAleh"):
                try:
                    sync_clone(candidate.parent)
                except Exception as error:
                    print("could not update the clone, using it as is:", error)
            return candidate.resolve()
    if IN_COLAB:
        target = Path("/content/SaloAleh")
        print("cloning", REPO_URL, "@", REPO_BRANCH)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(target)],
            check=True,
        )
        return (target / "DhikrSpeech").resolve()
    raise FileNotFoundError(
        "DhikrSpeech project not found. Set DHIKR_PROJECT_ROOT, or copy the "
        "DhikrSpeech folder to /content or to your Drive."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# A kernel that already imported src/ keeps the old modules even after the clone
# is updated, so drop them and let the imports below load the new code.
stale = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
for name in stale:
    del sys.modules[name]

for module_name, package in [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("soxr", "soxr"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
config.paths.ensure_dirs()

revision = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip()

print("project root :", PROJECT_ROOT)
print("code version :", revision or "(not a git checkout)")
print("config       :", CONFIG_PATH)
if stale:
    print("note         : reloaded %d cached src modules — re-run this notebook "
          "from the top so every stage uses the new code" % len(stale))
print()
print(config.summary())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


INFO src.config: loaded configuration from /content/SaloAleh/DhikrSpeech/configs/config.yaml


project root : /content/SaloAleh/DhikrSpeech
code version : 6856649 djir modle
config       : /content/SaloAleh/DhikrSpeech/configs/config.yaml
note         : reloaded 9 cached src modules — re-run this notebook from the top so every stage uses the new code

project root      : /content/drive/MyDrive/Dhikr Speech Dataset
classes           : phrases [6, 7] only
sample rate       : 16000 Hz, mono, PCM16
clip length       : 2 s (32000 samples)
features          : 40 log-mel bins x 197 frames (window 30 ms / hop 10 ms)
model             : ds_cnn (4 blocks x 64 filters)
training          : 300 epochs, batch 16, optimizer adam
augmentation      : on
seed              : 1337


# 01 · Dataset explorer

Reads the recordings on Drive and answers three questions before a single epoch is trained:

1. **How much data is there?** counts, per-class distribution, durations.
2. **Is any of it broken?** corrupted, empty, silent, stereo, wrong sample rate, duplicated.
3. **Is it balanced enough to train on?** thin classes are flagged.

Nothing is modified here — this notebook only reads. Cleaning happens in `02_preprocessing`.

Expected layout on Drive:

```
MyDrive/Dhikr Speech Dataset/
├── dataset/001/*.wav        one folder per phrase id
├── dataset/unknown/*.wav    filler / out-of-vocabulary audio
└── phrases.json             [{"id": 1, "text": "سبحان الله"}, ...]
```


## 1 · Locate the dataset

Paths come from `configs/config.yaml`. Change `paths.drive_root` / `paths.project_dir` there if
your dataset lives somewhere else — never edit paths in the notebooks.

In [27]:
from pathlib import Path

paths = config.paths
rows = [
    ("dataset", paths.dataset_path),
    ("phrases.json", paths.phrases_path),
    ("processed", paths.processed_path),
    ("checkpoints", paths.checkpoints_path),
    ("exports", paths.exports_path),
    ("logs", paths.logs_path),
    ("reports", paths.reports_path),
    ("noise (optional)", paths.noise_path),
]
for name, path in rows:
    print("%-18s %-6s %s" % (name, "ok" if Path(path).exists() else "MISSING", path))

if not paths.dataset_path.is_dir():
    raise FileNotFoundError(
        "dataset folder not found: %s\n"
        "Upload your recordings there, or point paths.drive_root / paths.project_dir "
        "at the right place in configs/config.yaml." % paths.dataset_path
    )


dataset            ok     /content/drive/MyDrive/Dhikr Speech Dataset/dataset
phrases.json       ok     /content/drive/MyDrive/Dhikr Speech Dataset/phrases.json
processed          ok     /content/drive/MyDrive/Dhikr Speech Dataset/processed
checkpoints        ok     /content/drive/MyDrive/Dhikr Speech Dataset/checkpoints
exports            ok     /content/drive/MyDrive/Dhikr Speech Dataset/exports
logs               ok     /content/drive/MyDrive/Dhikr Speech Dataset/logs
reports            ok     /content/drive/MyDrive/Dhikr Speech Dataset/reports
noise (optional)   MISSING /content/drive/MyDrive/Dhikr Speech Dataset/noise


## 2 · Phrases

`phrases.json` maps a class id to the Arabic phrase. Folder `001` is phrase id `1`.

In [28]:
import pandas as pd

from src.dataset import load_phrases

phrases = load_phrases(paths.phrases_path)
phrase_table = pd.DataFrame(
    [
        {
            "id": phrase.id,
            "folder": phrase.folder,
            "text": phrase.text,
            # False here means the phrase is excluded by classes.include_phrases,
            # so no clip of it reaches the manifest or the model.
            "trained": config.classes.selects(phrase.folder, paths.unknown_class),
            "folder exists": (paths.dataset_path / phrase.folder).is_dir(),
            "recordings": len(list((paths.dataset_path / phrase.folder).glob("*")))
            if (paths.dataset_path / phrase.folder).is_dir()
            else 0,
        }
        for phrase in phrases
    ]
)
print("%d phrases declared" % len(phrases))
if config.classes.enabled:
    print("classes.include_phrases restricts training to %s"
          % (config.classes.include_phrases,))
phrase_table


10 phrases declared
classes.include_phrases restricts training to [6, 7]


,id,folder,text,trained,folder exists,recordings
0,1,001,سبحان الله,False,True,35
1,2,002,الحمد لله,False,True,32
2,3,003,الله أكبر,False,True,33
3,4,004,لا إله إلا الله,False,True,28
4,5,005,أستغفر الله,False,True,28
5,6,006,سبحان الله وبحمده,True,True,35
6,7,007,سبحان الله العظيم وبحمده,True,True,32
7,8,008,لا حول ولا قوة إلا بالله,False,True,29
8,9,009,اللهم صل على محمد,False,True,33
9,10,010,اللهم صل وسلم على نبينا محمد,False,True,29


## 3 · Index every recording

Folders are the class vocabulary: numeric folders are phrases, `unknown` is the filler class that
teaches the model to stay quiet on everything else.

In [ ]:
from src.dataset import build_speaker_resolver, scan_dataset

# Who spoke each recording, resolved once for the whole dataset (speakers.csv, a
# per-speaker subfolder, or a filename convention - see split.speaker in the
# config). This is what lets the split keep a voice out of two splits, which is
# the difference between measuring "works for a new user" and "recognises these
# recordings again".
speaker_resolver = build_speaker_resolver(config)

index = scan_dataset(
    paths.dataset_path,
    phrases,
    unknown_class=paths.unknown_class,
    extensions=config.audio.file_extensions,
    classes=config.classes,
    speaker_resolver=speaker_resolver,
)

counts = index.counts()
count_table = pd.DataFrame(
    [
        {
            "class": label,
            "recordings": count,
            "share": count / max(len(index), 1),
            "speakers": len({s.speaker for s in index.samples if s.label == label and s.speaker}),
            "text": index.label_text(label),
        }
        for label, count in counts.items()
    ]
).sort_values("recordings", ascending=False)

print("recordings :", len(index))
print("classes    :", index.num_classes)
print("speakers   :", len(index.speakers()), "(source: %s)" % index.speaker_source)
if index.negative_counts():
    print("negatives  :", ", ".join("%s=%d" % item for item in index.negative_counts().items()))
count_table

## 4 · Speakers — who is in this dataset?

The single most consequential thing to know about the recordings. A model recognises a voice it has
heard far more easily than a stranger's, so a split that puts the same speaker in train and test
reports an accuracy that answers the wrong question — and the app is installed by people the model
has never heard.

Speaker ids come from `speakers.csv`, from a per-speaker subfolder (`dataset/006/ali/*.wav`), or from
a filename convention (`ali_001.wav`) — whichever the config selects. **When none of them applies,
this section says so prominently rather than letting the test split look like a measurement of
generalisation.**

In [ ]:
from src import visualization as viz

speakers = index.speaker_report()
print(speakers.summary(min_speakers=config.quality.prototype_speakers))

if speakers.recordings_per_speaker:
    figure = viz.plot_speaker_distribution(speakers.recordings_per_speaker)
    viz.save_figure(figure, paths.reports_path / "01_speaker_distribution.png")
    display(pd.DataFrame([
        {"class": label, "speakers": len(names), "who": ", ".join(names[:8])}
        for label, names in sorted(speakers.speakers_per_class.items())
    ]))

## 5 · Validate

Every file is opened and decoded. This is the slow cell — a few minutes for thousands of clips —
and it is what finds silent takes and duplicated uploads.

Set `DEEP = False` to only read file headers (fast, but skips silence and duplicate detection).

In [ ]:
from src.dataset import validate_dataset

DEEP = True

def progress(done: int, total: int, every: int = 200) -> None:
    """Print progress without flooding the notebook output."""
    if done == total or done % every == 0:
        print("  %d / %d" % (done, total), flush=True)

report = validate_dataset(index, config.audio, deep=DEEP, progress=progress)
print()
print(report.summary())


/content/SaloAleh/DhikrSpeech/src/audio.py:105: UserWarning: PySoundFile failed. Trying audioread instead.
  samples, _ = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/content/SaloAleh/DhikrSpeech/src/audio.py:105: UserWarning: PySoundFile failed. Trying audioread instead.
  samples, _ = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/content/SaloAleh/DhikrSpeech/src/audio.py:105: UserWarning: PySoundFile failed. Trying audioread instead.
  samples, _ = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/a

### Issues found

| kind | meaning | what to do |
|---|---|---|
| `corrupted` | the file cannot be decoded | delete or re-record |
| `empty` | zero-length file | delete |
| `silent` | RMS below `audio.silence_dbfs` | delete — it teaches the model nothing |
| `stereo` | more than one channel | harmless, notebook 02 downmixes it |
| `sample_rate` | not 16 kHz | harmless, notebook 02 resamples it |
| `too_short` / `too_long` | outside `audio.min_duration` / `max_duration` | review; usually a truncated take |
| `duplicate` | identical audio to another file | delete the copy — duplicates leak across the train/test split |

`corrupted`, `empty` and `silent` files are excluded from preprocessing automatically.

In [ ]:
issues = report.issues_dataframe()
if len(issues):
    display(issues.groupby("kind").size().rename("count").to_frame())
    display(issues.head(50))
else:
    print("no issues found")

unusable = report.unusable_paths()
print()
print("%d file(s) will be excluded from preprocessing" % len(unusable))
for path in unusable[:20]:
    print("  ", path)


## 6 · Statistics and distribution

In [ ]:
from src import visualization as viz
from src.quality import dataset_quality

# Durations come from the validation pass above, so this is one report over the
# same measurements rather than a second walk of the dataset.
durations_by_path = {item.path: item.duration for item in report.file_stats if item.ok}

quality = dataset_quality(
    index,
    speakers,
    config.quality,
    durations=durations_by_path,
    classes=config.classes,
    unknown_class=paths.unknown_class,
)
print(quality.summary())
print()
display(quality.to_dataframe())
quality.save(paths.reports_path)

figure = viz.plot_class_distribution(
    counts, highlight_below=config.quality.prototype_recordings_per_class
)
viz.save_figure(figure, paths.reports_path / "01_class_distribution.png")

durations = [item.duration for item in report.file_stats if item.ok and item.duration > 0]
figure = viz.plot_duration_histogram(
    durations,
    min_duration=config.audio.min_duration,
    max_duration=config.audio.max_duration,
)
viz.save_figure(figure, paths.reports_path / "01_duration_histogram.png")

## 7 · Listen to one sample per class

A quick ear check catches problems no validator can: the wrong phrase in a folder, a clipped
microphone, background speech.

In [ ]:
import numpy as np
from IPython.display import Audio, display

from src.audio import load_audio
from src.features import LogMelExtractor

rng = np.random.default_rng(config.seed)
extractor = LogMelExtractor(config.features, config.audio.sample_rate)
by_class = index.by_class()

previews, titles = [], []
for label in index.class_names:
    samples_for_class = by_class.get(label, [])
    if not samples_for_class:
        continue
    chosen = samples_for_class[int(rng.integers(len(samples_for_class)))]
    try:
        clip = load_audio(chosen.path, config.audio.sample_rate)
    except Exception as error:
        print("could not load", chosen.path, error)
        continue
    print("%-10s %s" % (label, chosen.path.name))
    display(Audio(clip, rate=config.audio.sample_rate))
    previews.append(extractor(clip))
    titles.append(label)

if previews:
    figure = viz.plot_feature_grid(previews, titles, hop_ms=config.features.hop_ms)
    viz.save_figure(figure, paths.reports_path / "01_class_previews.png")


## 8 · Save the validation report

Written to `reports/` on Drive:

* `validation_report.json` — statistics, issue counts, every issue
* `validation_report_files.csv` — one row per file (duration, rate, channels, RMS, content hash)

Notebook 02 reads this file to know which recordings to skip.

In [ ]:
written = report.save(paths.reports_path)
for kind, path in written.items():
    print("%-6s %s" % (kind, path))

print()
if report.is_clean:
    print("dataset is clean — continue with 02_preprocessing.ipynb")
else:
    print("%d issue(s) recorded." % len(report.issues))
    print("Delete the duplicates and silent takes, then re-run this notebook.")
    print("Everything else is handled automatically by 02_preprocessing.ipynb.")


# 02 · Preprocessing

Turns the raw recordings into the exact tensors the model trains on, and freezes the train/val/test
split so every later notebook sees the same data.

Each recording is:

1. decoded and resampled to **16 kHz mono**,
2. **silence trimmed** (`audio.trim`),
3. **loudness normalised** to a target RMS (`audio.normalize`),
4. **fitted to `audio.clip_seconds`** by padding or cropping,
5. written as **PCM16 WAV** under `processed/<class>/`.

The result is `processed/manifest.csv` — the single input of notebooks 03, 04 and 05.

Re-running is cheap: existing files are skipped unless `OVERWRITE = True`.

## 1 · Index the dataset and load the validation report

If `01_dataset.ipynb` has been run, its report tells us which files to exclude. Otherwise a quick
header-only validation is run here.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.dataset import (
    build_speaker_resolver, load_phrases, scan_dataset, validate_dataset,
)

paths = config.paths
phrases = load_phrases(paths.phrases_path)
index = scan_dataset(
    paths.dataset_path,
    phrases,
    unknown_class=paths.unknown_class,
    extensions=config.audio.file_extensions,
    classes=config.classes,
    speaker_resolver=build_speaker_resolver(config),
)
print("indexed %d recordings across %d classes" % (len(index), index.num_classes))
print("speakers: %d (source: %s)" % (len(index.speakers()), index.speaker_source))

report_path = paths.reports_path / "validation_report.json"
BLOCKING = {"corrupted", "empty", "silent"}

if report_path.is_file():
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    unusable = sorted({item["path"] for item in payload["issues"] if item["kind"] in BLOCKING})
    print("loaded validation report from %s" % report_path)
else:
    print("no validation report found — running a quick header-only validation")
    quick = validate_dataset(index, config.audio, deep=False)
    unusable = quick.unusable_paths()

print("%d recording(s) excluded as unusable" % len(unusable))


## 2 · Write conditioned copies

Output goes to `processed/` on Drive. Set `OVERWRITE = True` after changing any `audio.*` setting —
otherwise clips written with the old settings are kept.

In [ ]:
from src.dataset import preprocess_dataset

OVERWRITE = False

def progress(done: int, total: int, every: int = 200) -> None:
    """Print progress without flooding the notebook output."""
    if done == total or done % every == 0:
        print("  %d / %d" % (done, total), flush=True)

records, summary = preprocess_dataset(
    index,
    config,
    exclude=unusable,
    overwrite=OVERWRITE,
    progress=progress,
)
print()
print(summary.summary())
print("clips in manifest:", len(records))

if summary.failed:
    print()
    print("failed files are listed above and are absent from the manifest")


## 3 · Split into train / val / test

**Grouped by speaker whenever the recordings carry one**, so every recording of a voice lands in
exactly one split and the validation and test numbers describe a speaker the model has never heard.
Exact class ratios are impossible once whole speakers have to move together — a speaker with a third
of the recordings cannot be split 75/15/10 — so the ratios in `split.*` are targets and the guarantee
is the grouping.

With no speaker information it falls back to a per-class stratified split, which keeps the ratios
exact and guarantees nothing about speakers. The check below then reports that the evaluation is
**not** speaker-independent, instead of leaving it to be assumed.

In [ ]:
from src.dataset import assign_splits, save_manifest, split_counts, verify_manifest_splits

records = assign_splits(records, config.split, config.seed)
counts = split_counts(records)
print("split sizes:", counts)

frame = pd.DataFrame([
    {"class": record.label, "split": record.split} for record in records
])
pivot = frame.pivot_table(index="class", columns="split", aggfunc=len, fill_value=0)
display(pivot)

# Fails the notebook on leakage rather than warning: every number measured after
# this point - test accuracy, calibrated thresholds, the readiness verdict - is
# computed from these splits, and a leak makes all of them describe recognising a
# known voice. (split.speaker.require_disjoint: false accepts that knowingly.)
split_speakers = verify_manifest_splits(records, config.split)
print()
print(split_speakers.summary(min_speakers=config.quality.prototype_speakers))

manifest_path = save_manifest(records, paths.manifest_path)
print()
print("manifest written to", manifest_path)

## 4 · Preview the front-end

What the model actually sees: a `(frames, mel bins)` log mel spectrogram. Shape and framing come
from `features.*`, and this exact geometry is what the exported TFLite model expects.

In [ ]:
import numpy as np
from IPython.display import Audio, display

from src.audio import load_audio, read_wav
from src.features import LogMelExtractor
from src import visualization as viz

extractor = LogMelExtractor(config.features, config.audio.sample_rate)
frames, mel_bins, channels = config.input_shape
print("model input: %d frames x %d mel bins x %d channel" % (frames, mel_bins, channels))

rng = np.random.default_rng(config.seed)
previews, titles = [], []
for label in sorted({record.label for record in records}):
    for_class = [record for record in records if record.label == label]
    chosen = for_class[int(rng.integers(len(for_class)))]
    clip, _ = read_wav(chosen.resolve(paths.processed_path))
    previews.append(extractor(clip))
    titles.append("%s · %s" % (label, Path(chosen.path).name))

figure = viz.plot_feature_grid(previews[:9], titles[:9], hop_ms=config.features.hop_ms)
viz.save_figure(figure, paths.reports_path / "02_feature_previews.png")

example = records[int(rng.integers(len(records)))]
raw = load_audio(example.source_path, config.audio.sample_rate)
conditioned, _ = read_wav(example.resolve(paths.processed_path))
print()
print("before / after conditioning:", Path(example.source_path).name)
display(Audio(raw, rate=config.audio.sample_rate))
display(Audio(conditioned, rate=config.audio.sample_rate))
viz.save_figure(
    viz.plot_waveform(conditioned, config.audio.sample_rate, title="conditioned clip"),
    paths.reports_path / "02_conditioned_waveform.png",
)


## 5 · Preview the augmentation

Augmentation runs **on the fly during training only** — nothing here is written to Drive. This cell
shows what each transform does to one clip so the ranges in `augmentation.*` can be sanity checked
by eye and by ear.

If `noise/` on Drive is empty, background noise falls back to synthetic white/pink noise. Dropping
real room recordings in there is the single cheapest accuracy win for a phone-deployed model.

In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor, spec_augment

noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
print(noise_bank.report())
print()

augmentor = WaveformAugmentor(config.augmentation, config.audio, noise_bank)
base, _ = read_wav(records[0].resolve(paths.processed_path))

TRANSFORMS = ["background_noise", "pitch_shift", "speed_perturb", "gain", "time_shift", "reverb"]

# Force one transform at a time by disabling the others.
def only(name: str):
    single = config.with_overrides({
        "augmentation.%s.probability" % other: 0.0
        for other in TRANSFORMS if other != name
    }).with_overrides({
        "augmentation.%s.probability" % name: 1.0,
        "augmentation.%s.enabled" % name: True,
    })
    return WaveformAugmentor(single.augmentation, single.audio, noise_bank)

variants = [("original", base)]
for name in TRANSFORMS:
    variants.append((name, only(name)(base, np.random.default_rng(config.seed))))

# Listen, do not just look: augmentation that changes what the phrase *is* teaches
# the model the wrong thing, and a spectrogram will not tell you that a 2-semitone
# shift has turned one speaker into an implausible one.
for name, clip in variants:
    print(name)
    display(Audio(clip, rate=config.audio.sample_rate))

panels = [extractor(clip) for _, clip in variants]
labels = [name for name, _ in variants]
panels.append(spec_augment(
    extractor(base),
    config.with_overrides({"augmentation.spec_augment.probability": 1.0}).augmentation,
    np.random.default_rng(config.seed),
))
labels.append("spec_augment")

figure = viz.plot_feature_grid(panels, labels, hop_ms=config.features.hop_ms)
viz.save_figure(figure, paths.reports_path / "02_augmentation_previews.png")

if not config.augmentation.reverb.enabled:
    print()
    print("augmentation.reverb is off - the preview above forced it on to show what")
    print("it does. Turn it on in config.yaml if the app will be used across a room;")
    print("keep the decay short, a long tail smears the consonants that separate the")
    print("nested phrases from each other.")

## 6 · Global feature statistics (optional)

Only needed when `features.normalize` is `global`. The default, `per_example`, normalises each clip
on its own and needs no dataset statistics — which also makes the Android port simpler.

In [ ]:
from src.features import compute_global_stats

if config.features.normalize == "global":
    train_records = [record for record in records if record.split == "train"]
    sample_records = train_records[: min(len(train_records), 2000)]
    raw_extractor = LogMelExtractor(
        config.with_overrides({"features.normalize": "none"}).features,
        config.audio.sample_rate,
    )
    stats = compute_global_stats(
        raw_extractor(read_wav(record.resolve(paths.processed_path))[0])
        for record in sample_records
    )
    stats_path = stats.save(paths.processed_path / "feature_stats.json")
    print("global stats over %d clips:" % len(sample_records), stats)
    print("written to", stats_path)
else:
    print("features.normalize = %r — no global statistics needed" % config.features.normalize)


## 7 · Done

Written to Drive:

* `processed/<class>/*.wav` — conditioned 16 kHz mono PCM16 clips
* `processed/manifest.csv` — path, class, phrase id, split for every clip
* `reports/02_*.png` — front-end and augmentation previews

Continue with `03_training.ipynb`.

In [ ]:
print("processed clips :", len(records))
print("splits          :", split_counts(records))
print("manifest        :", paths.manifest_path)
print("processed root  :", paths.processed_path)


# 03 · Training

Trains the DS-CNN phrase spotter on the manifest written by `02_preprocessing`.

Enabled by the config, not by editing this notebook: TensorBoard, mixed precision, early stopping,
checkpointing, resume, class weights, label smoothing, LR schedule, automatic train/val split,
fixed seed, batch size, epochs and optimizer.

**Resuming.** Re-run this notebook with the same `RUN_NAME`; `BackupAndRestore` picks the run up at
the epoch it stopped at, optimizer state included. Set a new `RUN_NAME` to start fresh.

**Runtime → Change runtime type → GPU** before running, or training falls back to CPU.

## 1 · Seed, precision, device

In [ ]:
import tensorflow as tf

from src.trainer import configure_mixed_precision, set_global_seed

RUN_NAME = config.model.name  # change to start a separate run

# `training.resume: true` means re-running this notebook restores the previous
# run's weights, optimiser state and epoch counter. That is what you want after a
# Colab disconnect, and exactly what you do not want after changing a
# hyperparameter - the change would be applied on top of the old model and the
# printed history would splice both runs together. Set this to True whenever the
# config changed since the last run.
FRESH_START = False

set_global_seed(config.seed)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

mixed = configure_mixed_precision(config.training.mixed_precision)
print("TensorFlow  :", tf.__version__)
print("GPU         :", [gpu.name for gpu in gpus] or "none — training on CPU")
print("mixed float16:", mixed)
print("run name    :", RUN_NAME)


## 2 · Load the manifest

In [ ]:
import pandas as pd

from src.dataset import (
    class_names_from_manifest, filter_split, load_manifest, split_counts,
    verify_manifest_splits,
)

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

train_records = filter_split(records, "train")
val_records = filter_split(records, "val")
test_records = filter_split(records, "test")

if not train_records or not val_records:
    raise ValueError("training needs a non-empty train and val split — re-run 02_preprocessing")

# Checked again here, not only after preprocessing: this is the manifest the
# run is about to train on, and it may have been written by an older version.
speaker_split = verify_manifest_splits(records, config.split)
if not speaker_split.known:
    print("!! no speaker information in the manifest - validation and test")
    print("   accuracy below are NOT speaker-independent. See split.speaker.")

print("classes :", len(class_names), class_names)
print("splits  :", split_counts(records))

display(pd.DataFrame(
    [{"class": record.label, "split": record.split} for record in records]
).pivot_table(index="class", columns="split", aggfunc=len, fill_value=0))

# A split of a few clips per class cannot measure a model, and a handful of clips
# per class cannot train one. Say so here rather than after an hour of training.
val_per_class = len(val_records) / len(class_names)
train_per_class = len(train_records) / len(class_names)
print()
print("train clips / class : %.1f" % train_per_class)
print("val clips / class   : %.1f  (val accuracy moves in steps of %.2f)"
      % (val_per_class, 1.0 / max(len(val_records), 1)))
if val_per_class < 5 or train_per_class < 30:
    print()
    print("!! this dataset is too small to train or to measure a %d-class model."
          % len(class_names))
    print("   Aim for 50-100+ recordings per class from 10+ speakers for a first")
    print("   usable model. Below that, expect the model to collapse to a single")
    print("   class and validation accuracy to sit at chance (%.4f)."
          % (1.0 / len(class_names)))
    print("   The sanity check in section 6b still tells you whether the pipeline")
    print("   itself is correct, which is the useful thing to know meanwhile.")

## 3 · Input pipeline

Training clips are decoded once and cached, then re-augmented every epoch. Validation is never
augmented and never shuffled.

In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor
from src.dataset import make_tf_dataset
from src.features import FeatureStats, LogMelExtractor

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None

extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)
noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
augmentor = WaveformAugmentor(config.augmentation, config.audio, noise_bank)

train_dataset = make_tf_dataset(train_records, config, extractor, training=True, augmentor=augmentor)
val_dataset = make_tf_dataset(val_records, config, extractor, training=False)

features_batch, labels_batch = next(iter(train_dataset))
print("features:", features_batch.shape, features_batch.dtype)
print("labels  :", labels_batch.shape, labels_batch.dtype)
print("expected:", config.input_shape)
assert tuple(features_batch.shape[1:]) == config.input_shape


## 4 · Class weights

Balanced weights counteract an uneven dataset, so a class with 60 recordings still matters as much
as one with 600. Disable with `training.class_weights: false`.

In [ ]:
from src.dataset import compute_class_weights

class_weight = compute_class_weights(
    [record.class_index for record in train_records], len(class_names)
)
display(pd.DataFrame(
    [
        {
            "class": class_names[index],
            "train clips": sum(1 for record in train_records if record.class_index == index),
            "weight": round(weight, 3),
        }
        for index, weight in sorted(class_weight.items())
    ]
))
print("class weights enabled:", config.training.class_weights)


## 5 · Build the model

DS-CNN: a strided convolutional stem followed by depthwise separable blocks. Small enough for a
phone, and it quantises to INT8 without a Flex delegate.

In [ ]:
from src.models import build_model, capacity_report, model_summary_text

model = build_model(config.input_shape, len(class_names), config.model)
model.summary()

print()
print(capacity_report(model, len(train_records), config.model, len(class_names)))

(paths.checkpoints_path / RUN_NAME).mkdir(parents=True, exist_ok=True)
(paths.checkpoints_path / RUN_NAME / "model_summary.txt").write_text(
    model_summary_text(model), encoding="utf-8"
)

## 6 · TensorBoard

Run this before `fit` and the charts update live during training.

In [ ]:
LOG_DIR = paths.logs_path / RUN_NAME / "tensorboard"
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("log dir:", LOG_DIR)

try:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", "--logdir '%s'" % LOG_DIR)
except Exception as error:  # not running under IPython
    print("start TensorBoard manually:  tensorboard --logdir '%s'" % LOG_DIR)
    print(error)


## 6b · Sanity check — can this pipeline learn at all?

Before spending an hour on a real run, prove that the model can **memorise a handful of clips**.
A few dozen unaugmented recordings, a fresh copy of the model, a couple of hundred steps: training
accuracy has to go to ~1.0. It only has to overfit, so anything less means the fault is upstream of
the hyperparameters — the features, the labels, or the model itself.

This is the test to run first whenever a run sits at chance (0.10 for 10 classes) and the model
predicts one class for everything.


In [ ]:
from src.dataset import make_tf_dataset
from src.trainer import sanity_overfit, sanity_overfit_report

SANITY_CLIPS = 40          # clips to memorise, a few per class
SANITY_STEPS = 200         # optimiser steps to do it in
RUN_SANITY_CHECK = True

if RUN_SANITY_CHECK:
    # Take a stratified handful: at least one clip per class, otherwise the test
    # can pass on a subset that happens to be a single class.
    by_class = {}
    for record in train_records:
        by_class.setdefault(record.class_index, []).append(record)
    subset = []
    position = 0
    while len(subset) < min(SANITY_CLIPS, len(train_records)):
        added = False
        for index in sorted(by_class):
            if position < len(by_class[index]) and len(subset) < SANITY_CLIPS:
                subset.append(by_class[index][position])
                added = True
        if not added:
            break
        position += 1

    # training=False -> no augmentation. Memorising augmented clips is a different,
    # much harder test and not what we are asking here.
    sanity_dataset = make_tf_dataset(
        subset, config, extractor, training=False, batch_size=min(16, len(subset)), shuffle=False
    )
    sanity = sanity_overfit(model, sanity_dataset, steps=SANITY_STEPS)
    print("clips            :", len(subset))
    print(sanity_overfit_report(sanity, len(class_names)))
else:
    print("sanity check skipped")


## 7 · Train

Checkpoints, logs and the config snapshot are written to Drive as training runs, so an interrupted
Colab session loses nothing. Interrupting this cell and re-running it resumes the run.

In [ ]:
import math

from src.trainer import Trainer

steps_per_epoch = math.ceil(len(train_records) / config.training.batch_size)

trainer = Trainer(
    config=config,
    model=model,
    num_classes=len(class_names),
    steps_per_epoch=steps_per_epoch,
    run_name=RUN_NAME,
)

# FRESH_START is set in section 1. It must be True after any config change (and
# after changing classes.include_phrases, which changes the output width), so the
# new settings are not applied on top of the previous run's weights.
if FRESH_START:
    trainer.reset_run()

trainer.compile()

total_steps = steps_per_epoch * config.training.epochs
print("steps per epoch :", steps_per_epoch)
print("total steps     :", total_steps)
if total_steps < 2000:
    # Convergence follows gradient steps, not epochs. Under a couple of thousand,
    # a DS-CNN trained from scratch is still near its initialisation: the loss
    # falls a little each epoch and accuracy looks pinned near chance.
    print("                  !! under 2000 steps - expect an undertrained model.")
    print("                     Lower training.batch_size or raise training.epochs.")
print("epochs          :", config.training.epochs)
print("checkpoints     :", trainer.checkpoint_dir)
print("resume enabled  :", config.training.resume)
print("resuming a run  :", trainer.is_resuming)
print()

# val_size is the clip count, not the batch count: the summary uses it to report
# how coarse val_accuracy is, and to say so when the split is too small to measure
# a train/val gap.
artifacts = trainer.fit(
    train_dataset, val_dataset, class_weight=class_weight, val_size=len(val_records)
)
print()
print(artifacts.summary())


## 8 · Training curves

In [ ]:
from src import visualization as viz

figure = viz.plot_training_history(artifacts.history, title="run: %s" % RUN_NAME)
viz.save_figure(figure, paths.reports_path / ("03_history_%s.png" % RUN_NAME))

display(pd.DataFrame(artifacts.history).tail(10))


## 9 · Quick check on the validation split

A full evaluation with per-class metrics, confusion matrix and error analysis is
`04_evaluation.ipynb`. This is only a smoke check that the best checkpoint reloads and performs.

In [ ]:
from src.metrics import evaluate_model
from src.trainer import load_trained_model

best_model = load_trained_model(artifacts.best_model_path)
result = evaluate_model(
    best_model,
    val_dataset,
    class_names,
    paths=[record.path for record in val_records],
    confidence_threshold=config.evaluation.confidence_threshold,
)
print(result.summary())
print()
print("predicted class distribution (a healthy model spreads across all classes):")
for label, count in result.prediction_distribution().items():
    print("  %-10s %d" % (label, count))
print()
print("best checkpoint:", artifacts.best_model_path)
print("continue with 04_evaluation.ipynb")


# 04 · Evaluation

Scores the best checkpoint on a split it never trained on and writes every chart and table to
`reports/` on Drive.

Produced here: accuracy, precision, recall, F1 (macro and weighted), a confusion matrix, per-class
metrics, one-vs-rest ROC with macro AUC, a false-positive and false-negative breakdown, and a
confidence-threshold sweep for the on-device reject gate.

Which split is used comes from `evaluation.split` (default `test`).

## 1 · Load the model and the evaluation split

In [ ]:
import pandas as pd

from src.dataset import class_names_from_manifest, filter_split, load_manifest, make_tf_dataset
from src.features import FeatureStats, LogMelExtractor
from src.trainer import load_trained_model

RUN_NAME = config.model.name

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

split_name = config.evaluation.split
eval_records = filter_split(records, split_name)
if not eval_records:
    fallback = "val"
    print("split %r is empty — falling back to %r" % (split_name, fallback))
    split_name, eval_records = fallback, filter_split(records, fallback)
if not eval_records:
    raise ValueError("no clips to evaluate — re-run 02_preprocessing")

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None
extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)

# No shuffling and no augmentation: predictions line up with eval_records.
eval_dataset = make_tf_dataset(
    eval_records,
    config,
    extractor,
    training=False,
    batch_size=config.evaluation.batch_size,
    shuffle=False,
)

print("checkpoint :", checkpoint)
print("split      : %s (%d clips)" % (split_name, len(eval_records)))
print("classes    :", len(class_names))


## 2 · Predict and score

In [ ]:
from src.metrics import evaluate_model

result = evaluate_model(
    model,
    eval_dataset,
    class_names,
    paths=[record.path for record in eval_records],
    confidence_threshold=config.evaluation.confidence_threshold,
    # Carried from the manifest so a false positive can be attributed to a kind of
    # audio ("it fires on partial phrases") rather than left as a bare count.
    negative_types=[record.negative_type for record in eval_records],
    unknown_class=config.paths.unknown_class,
)
print(result.summary())

macro, weighted = result.averaged("macro"), result.averaged("weighted")
display(pd.DataFrame([
    {"average": "macro", **{key: round(value, 4) for key, value in macro.items()}},
    {"average": "weighted", **{key: round(value, 4) for key, value in weighted.items()}},
]))

## 3 · Per-class metrics

Sorted worst first — the classes at the top are the ones that need more recordings.

In [ ]:
from src import visualization as viz

per_class = result.per_class()
table = result.to_dataframe().sort_values("f1")
display(table)

figure = viz.plot_per_class_metrics(per_class)
viz.save_figure(figure, paths.reports_path / "04_per_class_metrics.png")

weak = table[table["f1"] < 0.8]
if len(weak):
    print("classes below 0.80 F1:", ", ".join(weak["label"].tolist()))
else:
    print("every class is at or above 0.80 F1")


## 3b · The confusions that decide this product

Three directions, and they are not equally expensive:

* **unknown → target** — firing at audio that is not a dhikr. Every one of these becomes a false
  count in the app.
* **target → unknown** — staying quiet on a real dhikr. A miss; the user sees it and repeats.
* **target A → target B** — counting the wrong phrase. The nested phrases live here
  (`سبحان الله وبحمده` ⊂ `سبحان الله العظيم وبحمده`), and a healthy top-line accuracy can hide most
  of its error in this row.

Then the same false positives grouped by the **kind** of negative audio, which turns "the model has
some false positives" into a data-collection instruction.

In [ ]:
directional = result.directional_confusions()
display(pd.DataFrame([
    {"direction": "unknown -> target (false counts)",
     "clips": directional["unknown_to_target"],
     "rate": round(directional["unknown_to_target_rate"], 4)},
    {"direction": "target -> unknown (misses)",
     "clips": directional["target_to_unknown"],
     "rate": round(directional["target_to_unknown_rate"], 4)},
    {"direction": "target -> other target (wrong phrase)",
     "clips": directional["target_to_target_total"],
     "rate": None},
]))
if directional["target_to_target"]:
    display(pd.DataFrame(directional["target_to_target"]))

by_type = result.by_negative_type()
if by_type:
    display(pd.DataFrame([
        {
            "negative type": item.negative_type,
            "clips": item.clips,
            "accepted as a dhikr": item.accepted,
            "false-positive rate": round(item.false_positive_rate, 4),
            "95% CI": "%.1f%%-%.1f%%" % tuple(value * 100 for value in item.interval()),
            "predicted as": ", ".join("%s=%d" % pair for pair in item.predicted_as.items()),
        }
        for item in by_type
    ]))
    figure = viz.plot_negative_type_false_positives(
        by_type, limit=config.readiness.max_hard_negative_fp_rate
    )
    viz.save_figure(figure, paths.reports_path / "04_negative_type_false_positives.png")
else:
    print("no negative categories in the manifest.")
    print()
    print("Organise the unknown folder into subfolders and re-run stage 02:")
    print("  dataset/unknown/hard_negative/   near-miss phrases (THE important one)")
    print("  dataset/unknown/partial_phrase/  incomplete utterances")
    print("  dataset/unknown/other_dhikr/     dhikr that are not the targets")
    print("  dataset/unknown/normal_speech/   ordinary Arabic speech")
    print("  dataset/unknown/noise/           rooms, traffic, TV")
    print("They all still train as one 'unknown' class - only the reporting changes.")

## 4 · Confusion matrix

Rows are the true class, columns the prediction. Bright cells off the diagonal are the phrase pairs
the model mixes up — usually phrases that share a leading word.

In [ ]:
matrix = result.confusion_matrix

figure = viz.plot_confusion_matrix(matrix, class_names, normalize=True,
                                   title="confusion matrix (row-normalised)")
viz.save_figure(figure, paths.reports_path / "04_confusion_matrix.png")

figure = viz.plot_confusion_matrix(matrix, class_names, normalize=False,
                                   title="confusion matrix (counts)")
viz.save_figure(figure, paths.reports_path / "04_confusion_matrix_counts.png")

confusions = result.top_confusions(config.evaluation.top_k_confusions)
if confusions:
    display(pd.DataFrame(confusions, columns=["true", "predicted", "count"]))
else:
    print("no off-diagonal errors")


## 5 · ROC

One-vs-rest per class. A class with no examples in this split is skipped, since its AUC is
undefined.

In [ ]:
if config.evaluation.roc:
    curves = result.roc_curves()
    macro_auc = curves.get("__macro__", {}).get("auc")
    print("macro AUC:", round(macro_auc, 4) if macro_auc is not None else "n/a")

    figure = viz.plot_roc_curves(curves)
    viz.save_figure(figure, paths.reports_path / "04_roc_curves.png")

    display(pd.DataFrame(
        [
            {"class": label, "auc": round(float(payload["auc"]), 4)}
            for label, payload in curves.items()
            if label != "__macro__"
        ]
    ).sort_values("auc"))
else:
    print("evaluation.roc is false — skipped")


## 6 · False positives and false negatives

* **False positive** — another phrase was predicted as this class. On device this is a phantom count.
* **False negative** — this class was said but predicted as something else. On device this is a missed count.

Listed for the classes with the most errors, with the file path so the clip can be listened to.

In [ ]:
from dataclasses import asdict

worst = sorted(per_class, key=lambda item: item.false_positives + item.false_negatives, reverse=True)
limit = config.evaluation.error_examples

for metrics in worst[:5]:
    if metrics.false_positives == 0 and metrics.false_negatives == 0:
        continue
    print("=" * 78)
    print("%s — %d false positive(s), %d false negative(s), support %d"
          % (metrics.label, metrics.false_positives, metrics.false_negatives, metrics.support))

    false_positives = result.false_positives(metrics.label, limit)
    if false_positives:
        print("\nfalse positives (predicted %s, actually something else):" % metrics.label)
        display(pd.DataFrame([asdict(case) for case in false_positives]))

    false_negatives = result.false_negatives(metrics.label, limit)
    if false_negatives:
        print("\nfalse negatives (%s said, predicted otherwise):" % metrics.label)
        display(pd.DataFrame([asdict(case) for case in false_negatives]))

if not any(item.false_positives or item.false_negatives for item in per_class):
    print("no errors on this split")


## 7 · Listen to the errors

The fastest way to tell a model problem from a data problem: if the clip sounds like the predicted
phrase, the label is wrong, not the model.

In [ ]:
from IPython.display import Audio, display

from src.audio import read_wav

errors = result.all_errors(limit=8)
if not errors:
    print("no misclassified clips to play")
for case in errors:
    print("true %-10s predicted %-10s confidence %.3f  %s"
          % (case.true_label, case.predicted_label, case.confidence, case.path))
    try:
        clip, _ = read_wav(paths.processed_path / case.path)
        display(Audio(clip, rate=config.audio.sample_rate))
    except Exception as error:
        print("  could not load:", error)


## 8 · Confidence threshold

On device the model runs continuously, so a prediction below a threshold should be discarded rather
than counted. This sweep picks that threshold: raise it until the error rate is acceptable, then
check how many correct detections it costs.

In [ ]:
import numpy as np

correct = result.y_true == result.y_pred
figure = viz.plot_confidence_distribution(
    result.confidence, correct, threshold=config.evaluation.confidence_threshold
)
viz.save_figure(figure, paths.reports_path / "04_confidence_distribution.png")

sweep = pd.DataFrame([
    result.rejection_stats(threshold) for threshold in np.arange(0.0, 1.0, 0.05)
])
sweep["accuracy_on_accepted"] = sweep["accuracy_on_accepted"].round(4)
sweep["accept_rate"] = sweep["accept_rate"].round(4)
display(sweep)

print()
print("current threshold (evaluation.confidence_threshold = %.2f):"
      % config.evaluation.confidence_threshold)
print(result.rejection_stats())


## 9 · Save the report

Written to `reports/`:

* `evaluation.json` — all metrics
* `evaluation_per_class.csv`
* `evaluation_errors.csv` — every misclassified clip
* `evaluation_confusion_matrix.csv`
* `04_*.png` — every chart above

In [ ]:
written = result.save(paths.reports_path)
for kind, path in written.items():
    print("%-18s %s" % (kind, path))

print()
print("charts:")
for path in sorted(paths.reports_path.glob("04_*.png")):
    print("  ", path)

print()
print("accuracy %.4f on the %s split — continue with 05_export.ipynb"
      % (result.accuracy, split_name))


# 05 · Export

Converts the trained checkpoint into the artefacts the Android app ships, then benchmarks and
verifies each one.

| variant | weights | activations | typical use |
|---|---|---|---|
| `float32` | float32 | float32 | reference — matches Keras exactly |
| `dynamic_range` | int8 | float32 | ~4× smaller, no calibration data needed |
| `int8` | int8 | int8 | smallest and fastest on phones; needs calibration clips |

Every variant is benchmarked (size, latency, arena estimate) and verified against the Keras model on
real clips, so a quantisation that damages accuracy is visible before it ships.

A checkpoint trained with mixed precision computes in float16, and TFLite has no float16 kernels for
`Conv2D` / `DepthwiseConv2dNative` / `Relu` — converting it directly fails with *"op is neither a
custom op nor a flex op"*. `export_all` handles this: it rebuilds the model in float32 first, which
is lossless because mixed precision keeps the master weights in float32 all along.


## 1 · Load the model and rebuild the front-end

In [ ]:
import numpy as np
import pandas as pd

from src.dataset import (
    class_names_from_manifest, filter_split, load_manifest, load_phrases, make_tf_dataset,
)
from src.features import FeatureStats, LogMelExtractor
from src.trainer import load_trained_model

RUN_NAME = config.model.name

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None
extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)

print("checkpoint :", checkpoint)
print("classes    :", len(class_names))
print("input shape:", config.input_shape)


## 2 · Calibration and verification clips

INT8 quantisation measures activation ranges from real features, so calibration comes from the
**training** split. Verification compares TFLite against Keras on **held-out** clips.

In [ ]:
from src.export import collect_features

train_records = filter_split(records, "train")
holdout_records = filter_split(records, "test") or filter_split(records, "val")

calibration_dataset = make_tf_dataset(
    train_records, config, extractor, training=False, batch_size=32, shuffle=False
)
calibration_features = collect_features(
    calibration_dataset, config.export.representative_samples
)

verification_features = None
if holdout_records:
    verification_dataset = make_tf_dataset(
        holdout_records, config, extractor, training=False, batch_size=32, shuffle=False
    )
    verification_features = collect_features(verification_dataset, 200)

# Quantisation damage is not uniform. A model can agree with Keras on 99% of clean
# test clips and disagree exactly on the near-miss audio that decides the
# false-activation rate, so the hard negatives are verified as their own subset.
extra_verification = {}
hard_negative_records = [
    record for record in records if record.negative_type == "hard_negative"
]
if hard_negative_records:
    extra_verification["hard negatives"] = collect_features(
        make_tf_dataset(
            hard_negative_records, config, extractor, training=False,
            batch_size=32, shuffle=False,
        ),
        200,
    )

print("calibration clips :", calibration_features.shape)
print("verification clips:", None if verification_features is None else verification_features.shape)
for name, features in extra_verification.items():
    print("%-18s: %s" % (name, features.shape))
if not hard_negative_records:
    print()
    print("no hard-negative clips in the manifest - the quantised model is only")
    print("verified on ordinary clips, which is the easy case.")

## 3 · Export, benchmark, verify

Conversion failures are isolated per variant: if INT8 fails, the other two are still produced.

In [ ]:
from src.export import export_all

metrics_payload = {}
evaluation_json = paths.reports_path / "evaluation.json"
if evaluation_json.is_file():
    import json

    payload = json.loads(evaluation_json.read_text(encoding="utf-8"))
    metrics_payload = {
        "accuracy": payload.get("accuracy"),
        "macro": payload.get("macro"),
        "num_samples": payload.get("num_samples"),
    }

phrases = {phrase.id: phrase.text for phrase in load_phrases(paths.phrases_path)}

bundle = export_all(
    model=model,
    config=config,
    class_names=class_names,
    frontend=extractor.metadata(),
    calibration_features=calibration_features,
    verification_features=verification_features,
    phrases=phrases,
    metrics=metrics_payload,
    extra_verification=extra_verification,
)

print()
print(bundle.table())


## 4 · Compare the variants

`expected_android_ms` is the measured latency multiplied by `export.android_latency_factor` — an
estimate for comparing variants, not a measurement. Measure on a real device before quoting it.

In [ ]:
from dataclasses import asdict

from src import visualization as viz

benchmarks = [item.benchmark for item in bundle.models if item.benchmark]
frame = pd.DataFrame([asdict(item) for item in benchmarks])
display(frame[[
    "name", "size_kb", "mean_latency_ms", "median_latency_ms", "p95_latency_ms",
    "arena_estimate_kb", "expected_android_ms", "input_dtype", "output_dtype",
]].round(3))

if benchmarks:
    figure = viz.plot_benchmark(benchmarks)
    viz.save_figure(figure, paths.reports_path / "05_benchmark.png")

verifications = [item.verification for item in bundle.models if item.verification]
if verifications:
    display(pd.DataFrame([asdict(item) for item in verifications]).round(5))
    for item in verifications:
        if not item.passed:
            print("WARNING: %s disagrees with the Keras model — do not ship it "
                  "without checking accuracy in 04_evaluation" % item.name)


## 5 · Front-end parameters for Android

The model takes log mel features, not raw audio, so the Android side must produce **identical**
features. These files pin that contract:

* `model_meta.json` — sample rate, clip length, FFT/window/hop, mel range, log offset, normalisation
* `mel_filterbank.json` — the exact mel matrix, so no filterbank has to be re-derived on device
* `labels.txt` — class order, one label per line
* `labels_phrases.json` — class index → phrase id → Arabic text

`README.md` has the matching Kotlin front-end.

In [ ]:
filterbank_path = extractor.save_filterbank(paths.exports_path / "mel_filterbank.json")
bundle.filterbank_path = filterbank_path

print("labels     :", bundle.labels_path)
print("metadata   :", bundle.metadata_path)
print("filterbank :", filterbank_path)
print()
for key, value in extractor.metadata().items():
    print("%-14s %s" % (key, value))


## 5b · Archive this export to history

`export_all` overwrites the export root every run, so the root always holds the **latest** model — the
one the app ships. This copies the whole export (the `.tflite` variants and their sidecars —
`labels.txt`, `model_meta.json`, `mel_filterbank.json`) into a dated snapshot under
`exports/history/<datetime>_<phrases>_<accuracy>/`, so every published model is kept and can be told
apart later. The bulky `saved_model/` is left out of snapshots.

The accuracy in the folder name is the one loaded above from `reports/evaluation.json` — re-run
**04 · Evaluation** after re-training so a snapshot is not stamped with a stale number (`accNA` means
no evaluation was found). The folder name is only a label; `model_meta.json` travels inside the
snapshot with the full metrics.

A Space pointed at the export root loads only the latest model — the fetcher skips `history/`. To
publish an older model, point the Space straight at its `history/<name>/` subfolder.


In [ ]:
from src.export import archive_export

# metrics_payload / evaluation_json come from the export cell above.
accuracy = metrics_payload.get("accuracy")
try:
    archive_dir = archive_export(
        paths.exports_path,
        include_phrases=config.classes.include_phrases,
        include_unknown=config.classes.include_unknown,
        accuracy=accuracy,
    )
except Exception as error:
    # The export root is already complete — the app ships from there. A failed
    # archive (e.g. a Google Drive FUSE copy hiccup) must not fail a good run.
    archive_dir = None
    print("could not archive this export to history:", error)

if archive_dir is not None:
    print("archived this export to:")
    print("  ", archive_dir)
    if accuracy is None:
        print("   !! accuracy unknown (accNA) — run 04 · Evaluation first so the "
              "snapshot folder is labelled with a real number")
    else:
        print("   accuracy %.4f from %s" % (accuracy, evaluation_json))
    print()
    for path in sorted(archive_dir.iterdir()):
        if path.is_file():
            print("    %-34s %8.1f KB" % (path.name, path.stat().st_size / 1024.0))


## 6 · What to ship

The recommendation is the smallest variant that still agrees with the Keras model. Override it if a
device measurement says otherwise.

In [ ]:
recommended = bundle.recommended()
if recommended is None:
    print("no variant passed verification — re-check the calibration clips and re-run")
else:
    print("recommended :", recommended.name)
    print("file        :", recommended.path)
    print("size        : %.2f MB" % (recommended.benchmark.size_kb / 1024.0))
    print("latency     : %.2f ms mean on this machine" % recommended.benchmark.mean_latency_ms)
    if recommended.verification:
        print("agreement   : %.2f%% with Keras" % (recommended.verification.agreement * 100))

from src.export import HISTORY_DIRNAME

print()
print("exports on Drive (the %s/ archive is listed by the cell above):" % HISTORY_DIRNAME)
history_dir = paths.exports_path / HISTORY_DIRNAME
for path in sorted(paths.exports_path.rglob("*")):
    if path.is_file() and history_dir not in path.parents:
        print("  %-34s %8.1f KB" % (path.name, path.stat().st_size / 1024.0))


## 7 · Android integration

Copy into `app/src/main/assets/`:

```
dhikr_int8.tflite      (or the recommended variant)
labels.txt
model_meta.json
mel_filterbank.json
```

On device, per inference:

1. record 16 kHz mono PCM16 into a ring buffer,
2. take the last `clip_seconds` of audio (`clip_samples` samples),
3. trim / normalise exactly as `model_meta.json.audio` describes,
4. compute the log mel spectrogram with the parameters in `model_meta.json.frontend`,
5. feed `(frames, n_mels, 1)` to the interpreter,
6. reject predictions below the threshold chosen in notebook 04, and debounce repeats so one spoken
   phrase counts once.

The full Kotlin front-end, the Gradle dependency and the threshold/debounce guidance are in
`README.md` under *Integrate into Android*.

In [ ]:
print("Export complete.\n")
print("copy to app/src/main/assets/:")
for name in ["labels.txt", "model_meta.json", "model_metadata.json", "mel_filterbank.json"]:
    path = paths.exports_path / name
    if path.exists():
        print("  ", path)
if recommended is not None:
    print("  ", recommended.path)

print()
print("model_metadata.json is the Android contract: tensor quantisation, window and")
print("hop, thresholds, detector parameters, model SHA256. Stage 06 rewrites it with")
print("CALIBRATED thresholds - until then it carries the configured defaults and says")
print("so in its own 'warnings' field.")
print()
print("Rejected variants:")
for name, reason in bundle.rejected() or [("none", "every variant agreed with Keras")]:
    print("  %-14s %s" % (name, reason))
print()
print("Growing the dataset later: add recordings to dataset/<class>/ and re-run stages 01 -> 05.")
print("Preprocessing skips clips it has already written, so re-runs only cost the new files.")

# 06 · Streaming — real-world evaluation

Everything up to here measured the model on **isolated clips**: given that someone said one of these
phrases, which one was it. The app asks something harder. The microphone is open, the user recites,
and the counter has to go up **exactly once per utterance** — while staying silent through
conversation, television, other dhikr and the rest of an ordinary room.

Clip accuracy cannot see any of the ways that goes wrong:

* one dhikr covers several windows, so counting windows counts it several times;
* confidence wobbles, so one utterance splits into two counts;
* every second of unrelated speech is another chance to fire.

So this stage runs the model the way it is deployed — a sliding window over continuous audio, then a
state machine (`idle → candidate → confirmed → cooldown`, with hysteresis and a per-class cooldown)
that turns windows into counted events — and measures **events**, not windows.

The number that decides whether this feature is usable is **false activations per hour**. A missed
dhikr is a small annoyance the user notices and repeats; a counter that ticks during a conversation
is a broken feature. The threshold is therefore chosen as *the lowest one that stays inside the
false-activation budget*, not the one that maximises accuracy.

**This stage needs long-form recordings**, which the clip dataset does not contain:

```
Dhikr Speech Dataset/streaming_test/
├── audio/
│   ├── session_001.wav      you reciting normally for a few minutes
│   ├── session_002.wav      another speaker, another room
│   ├── negative_tv.wav      television, no dhikr at all
│   └── negative_talk.wav    ordinary conversation, no dhikr at all
└── annotations.json
```

Without them the cells below explain what to record and skip cleanly — the notebook still runs top to
bottom. Nothing else in the pipeline depends on this stage, but **no model should ship without it**:
until it runs, the counter has never been measured as a counter.

## 6.1 · Streaming configuration

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.dataset import class_names_from_manifest, load_manifest
from src.features import FeatureStats, LogMelExtractor
from src.streaming import StreamingFrontend
from src.trainer import load_trained_model

RUN_NAME = config.model.name
paths = config.paths

records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None

# The same front-end the model was trained with - StreamingFrontend wraps the
# training conditioning and log-mel extractor rather than reimplementing them, so
# a window here is byte-identical to a clip there.
frontend = StreamingFrontend(config, stats=stats)
streaming = config.streaming
detector = streaming.detector

print("checkpoint     :", checkpoint)
print("classes        :", class_names)
print("window         : %.2f s (%d samples)" % (
    streaming.window_for(config.audio),
    int(streaming.window_for(config.audio) * config.audio.sample_rate)))
print("hop            : %.2f s -> %.1f windows/second" % (
    streaming.hop_seconds, 1.0 / streaming.hop_seconds))
print("smoothing      : %s" % (
    "%s over %d windows" % (streaming.smoothing.method, streaming.smoothing.window)
    if streaming.smoothing.enabled else "off"))
print()
print("detector")
print("  activation      : %.2f" % detector.confidence_threshold)
print("  release         : %.2f  (hysteresis: an ongoing event survives a dip)" % detector.release_threshold)
print("  consecutive hits: %d" % detector.min_consecutive_hits)
print("  cooldown        : %.0f ms (per class)" % detector.cooldown_ms)
print("  never counted   : %s" % ", ".join(detector.ignore_labels))
print()
print("FA/hour budget : %.2f" % streaming.target_false_activations_per_hour)
print()
print("These are STARTING POINTS, not tuned values. Section 6.8 calibrates them")
print("against the recordings below; until then treat them as arbitrary.")

## 6.2 · Load the long-form test audio

Each recording is scanned once and its per-window probabilities are kept. Every threshold in the
sweep later reuses them, so a twelve-point calibration costs twelve passes of a state machine rather
than twelve runs of the model.

In [ ]:
from src.audio import load_audio
from src.streaming import scan_signal
from src.streaming_eval import annotation_template, load_annotations

streaming_root = paths.streaming_path
audio_dir = streaming_root / streaming.audio_subdir
annotations_path = streaming_root / streaming.annotations_file

HAVE_STREAMING = annotations_path.is_file()
clips, scans, pairs = [], [], []

if not HAVE_STREAMING:
    found = sorted(p.name for p in audio_dir.glob("*.wav")) if audio_dir.is_dir() else []
    print("no streaming annotations at", annotations_path)
    print()
    print("What to record (phone microphone, the same way the app will hear it):")
    print("  1. A few minutes of yourself reciting normally, with natural pauses.")
    print("  2. The same from one or two other people, in different rooms.")
    print("  3. AT LEAST as much NEGATIVE audio: ordinary conversation, TV, radio,")
    print("     Quran recitation, other dhikr, a quiet room, the street. No target")
    print("     dhikr at all in these - they need no annotation and they carry the")
    print("     false-activation rate, which is the number that decides the feature.")
    print()
    print("Then write %s:" % annotations_path)
    print(json.dumps(annotation_template(found or ["session_001.wav", "negative_tv.wav"]),
                     indent=2))
    print()
    print("Timings need only be roughly right - matching is tolerant")
    print("(streaming.matching.tolerance_seconds = %.1f s)." % streaming.matching.tolerance_seconds)
    print()
    print("Sections 6.3-6.10 will skip until this exists.")
else:
    clips = [clip for clip in load_annotations(annotations_path, audio_dir) if clip.path.is_file()]
    for clip in clips:
        samples = load_audio(clip.path, config.audio.sample_rate)
        scan = scan_signal(samples, model, frontend, streaming, class_names, source=clip.name)
        scans.append(scan)
        pairs.append((clip, scan))

    total_seconds = sum(scan.duration for scan in scans)
    negative_seconds = sum(scan.duration for clip, scan in pairs if clip.is_negative)
    display(pd.DataFrame([
        {
            "file": clip.name,
            "minutes": round(scan.duration / 60.0, 2),
            "windows": scan.num_windows,
            "annotated dhikr": len(clip.events),
            "negative only": clip.is_negative,
            "type": clip.negative_type,
        }
        for clip, scan in pairs
    ]))
    print("total audio    : %.1f min (%.1f min with no dhikr in it)"
          % (total_seconds / 60.0, negative_seconds / 60.0))
    print("annotated dhikr: %d" % sum(len(clip.events) for clip in clips))
    if negative_seconds < 600:
        print()
        print("!! under 10 minutes of negative audio. FA/hour is extrapolated from what")
        print("   happened here, so at this length a single false activation is worth")
        print("   %.1f/hour - the number cannot be compared against a budget of %.2f."
              % (3600.0 / max(negative_seconds, 1.0),
                 streaming.target_false_activations_per_hour))

## 6.3 · Sliding-window predictions

The top panel is every class's probability over time with the activation and release thresholds drawn
in — this is where you see whether the hysteresis band sits in the right place. The bottom panel puts
what was annotated above what was detected, so a miss, a false activation and a duplicate look
different rather than being three numbers in a table.

In [ ]:
from src import visualization as viz

if HAVE_STREAMING and pairs:
    PREVIEW = 2  # recordings to plot
    for clip, scan in pairs[:PREVIEW]:
        events = scan.detect(detector)
        figure = viz.plot_streaming_timeline(
            scan, events=events, truth=clip.events, detector=detector,
            title="%s — %d detected, %d annotated" % (clip.name, len(events), len(clip.events)),
        )
        viz.save_figure(figure, paths.reports_path / ("06_timeline_%s.png" % Path(clip.name).stem))
else:
    print("skipped - no streaming recordings")

## 6.4 · Event detector output

Each row is one count the app would have made: where the run started and ended, when the counter
would have ticked (`trigger`), and how confident the model was at its peak. A dhikr spanning eight
windows is **one** row — that is the whole point of the state machine.

In [ ]:
if HAVE_STREAMING and pairs:
    rows = []
    for clip, scan in pairs:
        for event in scan.detect(detector):
            rows.append({"file": clip.name, **event.to_dict()})
    events_frame = pd.DataFrame(rows)
    display(events_frame.head(40))
    print("%d event(s) detected across %d recording(s)" % (len(rows), len(pairs)))
    if not rows:
        print()
        print("Nothing fired at all. Before touching the threshold, check section 6.3:")
        print("if the target probability never rises, the model does not recognise")
        print("these recordings - a different room or speaker from the training set.")
else:
    print("skipped - no streaming recordings")

## 6.5 · Event-level metrics

A detection matches an annotation when they overlap, or when the trigger lands within
`streaming.matching.tolerance_seconds` of it — hand annotation of a long recording is not
millisecond-accurate, and the detector fires part-way through a phrase rather than at its start.

Each annotation can be matched once. A second detection on an already-matched utterance is a
**duplicate**, not a false positive: same utterance counted twice is a detector problem (cooldown,
hysteresis), firing at audio nobody spoke into is a model problem (hard negatives, more speakers).
Keeping them apart is what makes the number actionable.

In [ ]:
from src.streaming_eval import evaluate_streaming

if HAVE_STREAMING and pairs:
    metrics = evaluate_streaming(pairs, detector, streaming.matching)
    print(metrics.summary(streaming.target_false_activations_per_hour))
    print()
    display(pd.DataFrame(metrics.per_file))
    written = metrics.save(paths.reports_path)
    print("written to", written)
else:
    metrics = None
    print("skipped - no streaming recordings")

## 6.6 · False activations per hour

The release-critical number, on its own, because it is the one that decides whether this ships.

Read the second row rather than the first when they differ: FA/hour over *all* audio is diluted by
the recitation sessions, where the user is deliberately saying dhikr. FA/hour over **negative-only**
audio is the honest estimate of a phone left listening in a room.

In [ ]:
if HAVE_STREAMING and metrics is not None:
    budget = streaming.target_false_activations_per_hour
    rows = [
        {
            "audio": "all recordings",
            "minutes": round(metrics.duration_seconds / 60.0, 1),
            "false activations": metrics.false_positives,
            "per hour": round(metrics.false_activations_per_hour, 3),
            "within budget": metrics.false_activations_per_hour <= budget,
        }
    ]
    if metrics.negative_duration_seconds > 0:
        rows.append({
            "audio": "negative only (no dhikr)",
            "minutes": round(metrics.negative_duration_seconds / 60.0, 1),
            "false activations": metrics.negative_false_positives,
            "per hour": round(metrics.negative_false_activations_per_hour, 3),
            "within budget": metrics.negative_false_activations_per_hour <= budget,
        })
    display(pd.DataFrame(rows))
    print("budget: %.2f false activations per hour" % budget)

    if metrics.per_negative_type:
        print()
        print("by kind of negative audio - this says WHAT breaks the detector:")
        display(pd.DataFrame([
            {"type": name, "minutes": round(payload["minutes"], 1),
             "false activations": int(payload["false_positives"]),
             "per hour": round(payload["per_hour"], 3)}
            for name, payload in sorted(metrics.per_negative_type.items())
        ]))
    if metrics.highest_false_confidence:
        print()
        print("highest confidence on a false activation: %.3f" % metrics.highest_false_confidence)
        print("Listen to that moment. Whatever it is, that is the audio to collect")
        print("more of as a hard negative.")
else:
    print("skipped - no streaming recordings")

## 6.7 · Negative-only stress test

Release-critical, and the cheapest data in this whole notebook to collect: audio with **no** target
dhikr in it needs no annotation, because the right answer for the entire recording is zero. Any
conversation, television, Quran recitation or street recording counts.

The predicted-window distribution below is the diagnostic. Almost every window should be won by
`unknown`; a few per cent leaking to a phrase is what a false activation is built from.

In [ ]:
from src.streaming_eval import negative_stress_test

negative_scans = [scan for clip, scan in pairs if clip.is_negative] if HAVE_STREAMING else []

if negative_scans:
    stress = negative_stress_test(negative_scans, detector)
    print(stress.summary(streaming.target_false_activations_per_hour))
    (paths.reports_path / "06_negative_stress.json").write_text(
        json.dumps(stress.to_dict(), indent=2, ensure_ascii=False), encoding="utf-8")
else:
    stress = None
    print("no negative-only recordings.")
    print()
    print("This is the single cheapest recording to collect and the most")
    print("release-critical: put the phone next to a conversation or a television")
    print("for ten minutes, save it as streaming_test/audio/negative_*.wav, and add")
    print('an entry with "events": [] to annotations.json. No annotation needed.')

## 6.8 · Threshold sweep and calibration

The default policy is deliberately **not** "best F1": F1 weighs a missed dhikr and a false count
equally, and here they are not equal. It takes the false-activation budget as a hard constraint and
maximises recall inside it.

If no threshold satisfies the budget, that is reported as a failure rather than resolved by picking
0.95 — a model that fires confidently on non-dhikr audio cannot be fixed by a threshold, and the
report says what to collect instead.

In [ ]:
from src.streaming_eval import calibrate_per_class, calibrate_threshold

calibration = per_class_calibration = None
per_class_thresholds = {}

if HAVE_STREAMING and pairs:
    calibration = calibrate_threshold(pairs, streaming)
    print(calibration.summary())
    print()
    display(calibration.to_dataframe())

    figure = viz.plot_threshold_sweep(
        calibration.points,
        target_false_activations_per_hour=streaming.target_false_activations_per_hour,
        chosen=calibration.threshold,
    )
    viz.save_figure(figure, paths.reports_path / "06_threshold_sweep.png")

    if streaming.calibration.per_class:
        print()
        print("per-class calibration (each phrase gets its own share of the budget -")
        print("a nested phrase needs a stricter threshold than a distinct one):")
        per_class_calibration = calibrate_per_class(
            pairs, streaming, class_names, base_threshold=calibration.threshold)
        # `outcome`, not `result`: stage 04 binds `result` to the clip evaluation
        # and section 6.11 reads it. A loop variable here would silently replace it.
        for label, outcome in sorted(per_class_calibration.items()):
            if outcome.threshold is not None:
                per_class_thresholds[label] = outcome.threshold
            print("  %-8s %s" % (
                label,
                ("%.2f" % outcome.threshold) if outcome.threshold is not None
                else "no viable threshold"))
            if outcome.reason:
                print("           !! %s" % outcome.reason)
else:
    print("skipped - no streaming recordings")
    print()
    print("Clip-level fallback: this sweeps isolated clips instead, which cannot")
    print("produce a false-activation RATE (clips have no hours). Use it as a")
    print("shortlist, never as a substitute - continuous speech contains far more")
    print("windows and far more near-misses than a clip set does.")

### Clip-level fallback

Only a shortlist. A model that rejects 99% of two-second unknown clips can still fire several times a
minute on continuous speech, because a stream contains vastly more windows and vastly more near
misses than a clip set does.

In [ ]:
from src.dataset import filter_split, make_tf_dataset
from src.metrics import predict_dataset
from src.streaming_eval import clip_threshold_sweep

eval_records = filter_split(records, config.evaluation.split) or filter_split(records, "val")
if eval_records:
    eval_dataset = make_tf_dataset(
        eval_records, config, frontend.extractor, training=False,
        batch_size=config.evaluation.batch_size, shuffle=False,
    )
    y_true, y_prob = predict_dataset(model, eval_dataset)
    display(pd.DataFrame(clip_threshold_sweep(
        y_true, y_prob, class_names,
        thresholds=streaming.calibration.thresholds(),
        unknown_class=paths.unknown_class,
        negative_types=[record.negative_type for record in eval_records],
    )))
else:
    print("no evaluation split - re-run stage 02")

## 6.9 · Recommended production configuration

`exports/model_metadata.json` is the contract the Android app implements: tensor shapes and
quantisation parameters, window and hop, the calibrated thresholds, every detector parameter, and the
model's SHA256.

Shipping the `.tflite` without it means the app invents its own operating point, and every number
this notebook measured stops applying. The file records whether the thresholds were calibrated, so a
default can never be mistaken for a measurement.

In [ ]:
from src.android import build_model_metadata, write_frontend_parity, write_model_metadata
from src.android import interpreter_specs
from src.streaming import make_interpreter

exported = sorted(paths.exports_path.glob("dhikr_*.tflite"))
preferred = next((p for p in exported if p.name.endswith("int8.tflite")), None) or (
    exported[0] if exported else None)

tensors = {}
if preferred is not None:
    interpreter = make_interpreter(preferred)
    interpreter.allocate_tensors()
    tensors = interpreter_specs(interpreter)

payload = build_model_metadata(
    config,
    class_names,
    frontend.extractor.metadata(),
    model_path=preferred,
    tensors=tensors,
    thresholds=per_class_thresholds or None,
    global_threshold=calibration.threshold if calibration else None,
    calibration=calibration.to_dict() if calibration else None,
    metrics=metrics.to_dict() if metrics is not None else None,
)
metadata_path = write_model_metadata(paths.exports_path / "model_metadata.json", payload)

# The front-end parity fixture: a fixed WAV plus the exact feature tensor this
# pipeline produces from it. Android compares against a number instead of a
# description - a window length off by one sample costs accuracy silently.
parity = write_frontend_parity(paths.exports_path, frontend.extractor, config)

print("metadata   :", metadata_path)
for name, path in parity.items():
    print("parity %-4s: %s" % (name, path))
print()
print(json.dumps({
    "detector": payload["detector"],
    "streaming": payload["streaming"],
    "tensors": payload["tensors"],
}, indent=2, ensure_ascii=False)[:1600])
for warning in payload.get("warnings", []):
    print()
    print("!!", warning)

## 6.10 · TFLite streaming verification

Agreement on clean test clips is the easy case. What decides whether a quantised model is shippable
is whether it still agrees **on the windows a stream is made of** — and, above all, whether INT8 adds
false activations. A model that saves 4× in size and doubles the false-count rate is not a smaller
model, it is a worse product.

In [ ]:
from src.streaming_eval import evaluate_streaming as evaluate_events

tflite_metrics = tflite_stress = None
window_agreement = extra_false_per_hour = None

if HAVE_STREAMING and pairs and preferred is not None:
    quantised_pairs = []
    for clip, scan in pairs:
        samples = load_audio(clip.path, config.audio.sample_rate)
        quantised_pairs.append(
            (clip, scan_signal(samples, preferred, frontend, streaming, class_names,
                               source=clip.name))
        )

    agreements = []
    for (_, reference), (_, quantised) in zip(pairs, quantised_pairs):
        agreements.append(
            float((reference.probabilities.argmax(axis=1)
                   == quantised.probabilities.argmax(axis=1)).mean())
        )
    window_agreement = float(np.mean(agreements)) if agreements else None

    tflite_metrics = evaluate_events(quantised_pairs, detector, streaming.matching)
    extra_false_per_hour = (
        tflite_metrics.false_activations_per_hour - metrics.false_activations_per_hour
    )

    display(pd.DataFrame([
        {"model": "keras (float32)", "recall": round(metrics.recall, 4),
         "precision": round(metrics.precision, 4),
         "FA/hour": round(metrics.false_activations_per_hour, 3)},
        {"model": preferred.name, "recall": round(tflite_metrics.recall, 4),
         "precision": round(tflite_metrics.precision, 4),
         "FA/hour": round(tflite_metrics.false_activations_per_hour, 3)},
    ]))
    print("per-window class agreement: %.2f%%" % (window_agreement * 100.0))
    print("extra false activations   : %+.3f per hour" % extra_false_per_hour)
    limit = config.readiness.max_tflite_extra_false_activations_per_hour
    if extra_false_per_hour > limit:
        print()
        print("!! REJECT this variant for production: quantisation adds %.3f false"
              % extra_false_per_hour)
        print("   activations per hour (limit %.3f). Ship dynamic_range instead, or" % limit)
        print("   re-run INT8 calibration with representative features.")
elif preferred is None:
    print("no exported .tflite found - run stage 05 first")
else:
    print("skipped - no streaming recordings")

## 6.11 · Production readiness

The verdict, and what it is based on. Every check prints the measurement *and* the configured
threshold it was compared against, so "READY" always says what it means by ready.

A check that could not be measured is never a pass. A model with no streaming evaluation has not been
shown to count correctly — it has only failed to be shown wrong, and that reads `EXPERIMENTAL`.

In [ ]:
from src.dataset import manifest_speaker_report
from src.quality import dataset_quality
from src.readiness import build_readiness_report

speaker_summary = manifest_speaker_report(records)

quality_report = None
quality_json = paths.reports_path / "dataset_quality.json"
try:
    from src.dataset import load_phrases, scan_dataset, build_speaker_resolver

    index = scan_dataset(
        paths.dataset_path, load_phrases(paths.phrases_path),
        unknown_class=paths.unknown_class, extensions=config.audio.file_extensions,
        classes=config.classes, speaker_resolver=build_speaker_resolver(config),
    )
    quality_report = dataset_quality(
        index, speaker_summary, config.quality,
        classes=config.classes, unknown_class=paths.unknown_class,
    )
except Exception as error:
    print("dataset not reachable for the quality section:", error)

# Stage 04 leaves its EvaluationResult in `result`. Checked by shape rather than
# by name so an unrelated variable of the same name can never be read as one.
clip_result = globals().get("result")
if not hasattr(clip_result, "directional_confusions"):
    clip_result = None
    print("no clip evaluation in memory - run stage 04 for the clip section")

report = build_readiness_report(
    config.readiness,
    speakers=speaker_summary,
    quality=quality_report,
    clip_result=clip_result,
    streaming=metrics,
    stress=stress,
    calibration=calibration,
    per_class_thresholds=per_class_thresholds,
    tflite_verifications=[
        verification
        for item in globals().get("bundle").models
        for verification in item.all_verifications
    ] if hasattr(globals().get("bundle"), "models") else None,
    tflite_extra_false_activations=extra_false_per_hour,
)
print(report.summary())
print()
print("written to", report.save(paths.reports_path))

# 07 · Experiment — one model per dhikr?

A recurring proposal: instead of one model that classifies every phrase, train
**one binary detector per dhikr**, each specialised in its own phrase. This
section settles it by building both and scoring them on the same clips.

It is optional and it is not cheap: it trains one model per phrase, so with four
phrases it costs four training runs on top of the one you already have. Nothing
here touches the exported model — run it when you want the answer, skip it
otherwise.

**What gets compared.** Three questions, because they can disagree:

| Question | Metric | Why it matters |
|---|---|---|
| Can it detect *this* phrase? | ROC-AUC / average precision per phrase | Threshold-free, so neither side wins on tuning |
| Can it name the *right* phrase? | accuracy on phrase clips | Where nested phrases decide it: `سبحان الله` ⊂ `سبحان الله وبحمده` ⊂ `سبحان الله العظيم وبحمده` |
| Can it stay quiet on non-dhikr? | accept rate on `unknown` clips | A committee of binaries has no `unknown` output, only a threshold |

**What is held fixed** so the result is about the approach and not the setup: the
same manifest and splits, the same architecture, the same augmentation, the same
optimiser, the same seed and the same epoch budget. The evaluation dataset is
built once and every model runs over those same tensors.

The committee of binaries predicts by `argmax` over the per-model positive
scores — the deployment shape the proposal implies — and the multi-class model is
scored by `argmax` over the same phrases, so both choose from the same answers.

## 1 · Load the manifest, the trained model and the front-end

In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor
from src.dataset import class_names_from_manifest, filter_split, load_manifest
from src.experiments import phrase_labels
from src.features import FeatureStats, LogMelExtractor
from src.trainer import load_trained_model

RUN_NAME = config.model.name

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
multiclass_model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None
extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)

# The binary models must be trained with the same augmentation the multi-class
# model got, or the comparison measures the augmentation instead of the approach.
noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
augmentor = WaveformAugmentor(config.augmentation, config.audio, noise_bank)

targets = phrase_labels(class_names, paths.unknown_class)
eval_records = filter_split(records, config.evaluation.split) or filter_split(records, "val")

print("checkpoint :", checkpoint)
print("phrases    : %d %s" % (len(targets), targets))
print("eval clips :", len(eval_records))
print()
print("this will train %d model(s) of %d epochs each"
      % (len(targets), config.training.epochs))
if len(targets) < 2:
    print()
    print("!! only one phrase is in the vocabulary, so there is nothing to tell apart.")
    print("   The per-phrase detection numbers still work; the discrimination test")
    print("   will be skipped. Widen classes.include_phrases for the full comparison.")

## 2 · Train one detector per phrase and compare

Each run is written under `checkpoints/ovr_<phrase>/`, separate from the
multi-class run, and starts from scratch (`fresh=True`) so a detector left over
from a previous experiment cannot bleed into this one.

`EPOCHS = None` gives every binary model the same budget the multi-class model
had, which is what keeps the comparison fair. Shorten it only to rehearse the
mechanics — a shortened run tells you the code works, not which approach wins.

In [ ]:
from src.experiments import compare_one_vs_rest

EPOCHS = None  # None = the same budget as the multi-class run (keep it that way)

report, detectors = compare_one_vs_rest(
    config,
    records,
    multiclass_model,
    extractor,
    augmentor=augmentor,
    phrases=targets,
    epochs=EPOCHS,
    fresh=True,
    verbose=0,
    progress=lambda label, position, total: print(
        "[%d/%d] training one-vs-rest detector for %s ..." % (position, total, label)
    ),
)

print()
print(report.summary())

## 3 · Per-phrase detection

One row per phrase. Both columns answer the same question — how well the
approach separates that phrase from everything else — so the delta is the whole
story. Positive delta means the specialised binary model won.

These are threshold-free (AUC and average precision), so a difference here is a
difference in what the model learned, not in how it was tuned. Average precision
is the one to trust when a phrase has few clips: it is the more honest of the two
under heavy imbalance.

In [ ]:
display(report.to_dataframe())

print("mean AUC — multi-class : %.4f" % report.mean_auc_multiclass)
print("mean AUC — one-vs-rest : %.4f" % report.mean_auc_binary)
print()
for item in report.per_phrase:
    print("%-10s %3d clips | binary trained %d epochs, best val %s"
          % (item.label, item.support, item.binary_epochs,
             "n/a" if item.binary_val_accuracy is None
             else "%.4f" % item.binary_val_accuracy))

## 4 · Telling the phrases apart

Restricted to clips that really are a phrase, and to the phrase columns on both
sides — so `unknown` cannot absorb a mistake for the multi-class model, and the
committee is not asked a question it has no output for.

This is the test the nested prefixes decide. A binary detector for
`سبحان الله وبحمده` is never shown that `سبحان الله العظيم وبحمده` is a
*different* phrase; the multi-class model is trained on exactly that boundary. If
one-vs-rest is going to lose anywhere, it loses here — look at the confusion
pairs, not just the accuracy.

In [ ]:
import pandas as pd

if report.discrimination is None:
    print("skipped — fewer than two phrases in the vocabulary")
else:
    discrimination = report.discrimination
    intervals = discrimination.intervals()
    display(pd.DataFrame([
        {
            "approach": "multi-class (one softmax)",
            "correct": discrimination.multiclass_correct,
            "clips": discrimination.num_clips,
            "accuracy": round(discrimination.multiclass_accuracy, 4),
            "95% CI": "%.3f–%.3f" % intervals["multiclass"],
        },
        {
            "approach": "committee of %d binaries" % discrimination.num_phrases,
            "correct": discrimination.committee_correct,
            "clips": discrimination.num_clips,
            "accuracy": round(discrimination.committee_accuracy, 4),
            "95% CI": "%.3f–%.3f" % intervals["committee"],
        },
    ]))

    for title, pairs in (
        ("multi-class", discrimination.multiclass_confusions),
        ("committee", discrimination.committee_confusions),
    ):
        print()
        print("%s confusions:" % title)
        if not pairs:
            print("  none")
        for true_label, predicted_label, count in pairs:
            print("  %s heard as %s: %d" % (true_label, predicted_label, count))

## 5 · Staying quiet on non-dhikr audio

Both sides are read by the same rule: a clip is *accepted* when the highest
phrase score clears the threshold. The multi-class model's `unknown` output is
deliberately not consulted, so neither approach is judged by a mechanism the
other lacks.

Read the two numbers per approach as a pair. A false-accept rate alone flatters
whichever model fires less often — the recall column is what makes it a
comparison. This is one operating point and the two approaches produce
differently-calibrated scores, so treat it as a sanity check rather than the
verdict.

In [ ]:
if report.rejection is None:
    print("skipped — the vocabulary has no `unknown` class.")
    print("Train with classes.include_unknown and an `unknown` dataset folder to")
    print("measure open-set behaviour; without it neither approach can say")
    print("\"that was not a dhikr\".")
else:
    rejection = report.rejection
    display(pd.DataFrame([
        {
            "approach": "multi-class",
            "fires on unknown": "%d/%d" % (rejection.multiclass_unknown_accepted,
                                           rejection.unknown_clips),
            "false accept": round(rejection.multiclass_false_accept, 4),
            "accepts real phrases": round(rejection.multiclass_recall, 4),
        },
        {
            "approach": "committee",
            "fires on unknown": "%d/%d" % (rejection.committee_unknown_accepted,
                                           rejection.unknown_clips),
            "false accept": round(rejection.committee_false_accept, 4),
            "accepts real phrases": round(rejection.committee_recall, 4),
        },
    ]))
    print("threshold:", rejection.threshold)

## 6 · Save the comparison

Written next to the other reports. Keep it: the next time the "one model per
dhikr" question comes up, this file is the answer for the dataset you had on the
day, and re-running it after the dataset grows is how you find out whether the
answer changed.

In [ ]:
written = report.save(paths.reports_path)
print("saved:", written)
print()
for note in report.verdict():
    print("!!", note)
    print()

print("checkpoints left behind (delete them once you are done — they are not")
print("shipped and each one is a full model):")
for detector in detectors:
    print("  ", paths.checkpoints_path / detector.run_name)